# 🧱 Lab: Runtime Failure Debugging

**Module: Delivery: CLI, API, and Debugging Lab 2** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Diagnose** runtime-only failures that pass all tests but break when the application actually runs
2. **Apply** a debugging decision procedure to systematically narrow down failure causes
3. **Fix** missing environment variable errors using layered configuration with safe defaults
4. **Resolve** relative-path bugs that work in one directory but break from another
5. **Detect** port conflicts and implement a recovery strategy

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Command-Line Interface | A CLI contract defines how a command is used and how it behaves: its arguments and flags, what `--help` promises, which exit codes mean success or failure, and what goes to stdout versus stderr. | mini-build-cli |
| Layered Configuration | Defaults live in code, environments override them at deploy time, and secrets never go in Git, so you can change behavior safely without editing source. | mini-build-cli |
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. | mini-define-ai-protocol |
| Boundary Validation | Check user input and model output, the parts you do not control, so errors, abuse, and surprises are caught before they hit your core logic. | mini-boundary-schemas |
| HTTP Endpoint Contract | The promise an endpoint makes: which method and route it accepts, what request body is valid, and which status codes and JSON body come back. | This lab |
| Debugging Decision Procedure | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. | This lab |
| Filesystem Paths | Use Path objects, not strings, to build file locations that work across OSes. Prefer relative paths anchored to your project (not your shell), since relative paths depend on the current working directory. | This lab |

## Prerequisites

- **mini-build-cli**: Building CLIs with argparse, exit codes, stdout/stderr, layered configuration
- **mini-define-ai-protocol**: Interface seams that keep the core logic free of vendor and transport details
- **mini-boundary-schemas**: Validating inputs you do not control at the boundary

## Why "Green Tests, Broken Runtime" Happens

Your tests can all pass and the app can still crash when you run it for real. Tests run in a controlled environment: env vars are set, `cwd` is the project root, and ports are free. Real machines do not promise any of that. In this lab we'll build a small app with **three planted runtime bugs** and debug each one.

## Setup

In [1]:
import subprocess, sys, os, time, json
from pathlib import Path

from IPython.display import display, Markdown

def md(text):
    display(Markdown(text))

def run(cmd, env=None, cwd=None):
    # cmd is a list of arguments, e.g. [sys.executable, "cli.py", "Buy groceries"].
    # A list avoids shell quoting problems: each item reaches the program as one argument.
    merged_env = {**os.environ, **(env or {})}
    result = subprocess.run(
        cmd, capture_output=True, text=True,
        env=merged_env, cwd=cwd,
    )
    if result.stdout:
        print("[stdout]", result.stdout, end="")
    if result.stderr:
        print("[stderr]", result.stderr, end="")
    print(f"[exit code] {result.returncode}")
    return result

In [2]:
!pip install -q fastapi uvicorn httpx


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## The Debugging Decision Procedure

```
1. REPRODUCE   → Run the exact command that fails. Note the error message.
2. CLASSIFY    → What category is this? (env var? path? port? permission?)
3. ISOLATE     → Strip away everything else. Can you trigger the bug in one line?
4. FIX         → Apply the smallest change that resolves the issue.
5. VERIFY      → Re-run the original failing command. Does it pass now?
```

## Build the Buggy Application

We'll create a project directory with a small note-taking CLI + API app that has planted runtime bugs.

In [3]:
# Create project directory
project_dir = Path("debug_app")
project_dir.mkdir(exist_ok=True)
(project_dir / "data").mkdir(exist_ok=True)

print("Project structure created.")

Project structure created.


In [4]:
%%writefile debug_app/config.json
{"app_name": "NoteKeeper", "max_note_length": 500}

Writing debug_app/config.json


In [5]:
%%writefile debug_app/core.py
"""Core logic — no CLI or HTTP dependencies."""
import json
from pathlib import Path
from datetime import datetime, timezone


def load_config():
    # BUG 2: relative path — works only when cwd is the project dir
    config_path = Path("config.json")
    with open(config_path, encoding="utf-8") as f:
        return json.load(f)


def save_note(text: str, api_key: str, data_dir: Path) -> dict:
    config = load_config()
    if len(text) > config["max_note_length"]:
        raise ValueError(f"Note exceeds {config['max_note_length']} chars")

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    filename = f"note_{timestamp}.txt"
    (data_dir / filename).write_text(text, encoding="utf-8")

    # "app" comes from config.json, so a successful save proves the config was found
    return {"status": "saved", "file": filename, "app": config["app_name"]}

Writing debug_app/core.py


In [6]:
%%writefile debug_app/cli.py
import argparse
import json
import os
import sys
from pathlib import Path

from core import save_note


def main():
    parser = argparse.ArgumentParser(description="Save a note.")
    parser.add_argument("text")
    parser.add_argument("--data-dir", default="data")
    args = parser.parse_args()

    # BUG 1: crashes if NOTE_API_KEY is not set — no default, no helpful error
    api_key = os.environ["NOTE_API_KEY"]

    data_dir = Path(args.data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    try:
        print(json.dumps(save_note(args.text, api_key, data_dir), indent=2))
    except ValueError as e:
        print(f"error: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()

Writing debug_app/cli.py


In [7]:
%%writefile debug_app/api.py
"""HTTP interface seam."""
import os
from pathlib import Path

from fastapi import FastAPI
from pydantic import BaseModel, Field

from core import save_note

app = FastAPI(title="NoteKeeper API")


class NoteRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=500)


# BUG 3: hardcoded port 8000 in docs/examples — will conflict if already in use
# (The port bug is in how we *launch* the server, not in this file.)


@app.post("/note")
def create_note(req: NoteRequest):
    # BUG 1 (same pattern): crashes if NOTE_API_KEY not set
    api_key = os.environ["NOTE_API_KEY"]
    data_dir = Path("data")
    data_dir.mkdir(parents=True, exist_ok=True)
    return save_note(req.text, api_key, data_dir)

Writing debug_app/api.py


---

## Bug 1: Missing Environment Variable

### Step 1 — REPRODUCE

In [8]:
# Run the CLI on a "fresh machine": an env without NOTE_API_KEY
clean_env = {k: v for k, v in os.environ.items() if k != "NOTE_API_KEY"}
result = run(
    [sys.executable, "cli.py", "Buy groceries"],
    env=clean_env,
    cwd="debug_app",
)

[stderr] Traceback (most recent call last):
  File "C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-runtime-failure-debugging\debug_app\cli.py", line 30, in <module>
    main()
  File "C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-runtime-failure-debugging\debug_app\cli.py", line 17, in main
    api_key = os.environ["NOTE_API_KEY"]
              ~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "<frozen os>", line 714, in __getitem__
KeyError: 'NOTE_API_KEY'
[exit code] 1


### Step 2 — CLASSIFY

`KeyError: 'NOTE_API_KEY'` means this is an **environment variable** bug.

### Step 3 — ISOLATE

The failing line in `cli.py`:
```python
api_key = os.environ["NOTE_API_KEY"]  # raises KeyError when missing
```

### Step 4 — FIX

Use **layered configuration**: CLI arg → env var → helpful error.

In [9]:
%%writefile debug_app/cli.py
"""CLI — Bug 1 FIXED."""
import argparse
import json
import os
import sys
from pathlib import Path

from core import save_note


def resolve_api_key(cli_value):
    # Layer 1: an explicit CLI flag wins
    if cli_value is not None:
        return cli_value
    # Layer 2: the environment; .get() returns None instead of crashing
    env_value = os.environ.get("NOTE_API_KEY")
    if env_value is not None:
        return env_value
    # Layer 3: nothing found, so say how to fix it
    print("error: NOTE_API_KEY not set. Pass --api-key or set NOTE_API_KEY.", file=sys.stderr)
    sys.exit(1)


def main():
    parser = argparse.ArgumentParser(description="Save a note.")
    parser.add_argument("text")
    parser.add_argument("--data-dir", default="data")
    parser.add_argument("--api-key", default=None)
    args = parser.parse_args()

    api_key = resolve_api_key(args.api_key)

    data_dir = Path(args.data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)

    try:
        print(json.dumps(save_note(args.text, api_key, data_dir), indent=2))
    except ValueError as e:
        print(f"error: {e}", file=sys.stderr)
        sys.exit(1)


if __name__ == "__main__":
    main()

Overwriting debug_app/cli.py


### Step 5 — VERIFY

In [10]:
# With NOTE_API_KEY set, the CLI saves the note
run(
    [sys.executable, "cli.py", "Buy groceries"],
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
    cwd="debug_app",
)

[stdout] {
  "status": "saved",
  "file": "note_20261018_202215.txt",
  "app": "NoteKeeper"
}
[exit code] 0


CompletedProcess(args=['C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe', 'cli.py', 'Buy groceries'], returncode=0, stdout='{\n  "status": "saved",\n  "file": "note_20261018_202215.txt",\n  "app": "NoteKeeper"\n}\n', stderr='')

Now trigger the path bug by running the CLI from outside `debug_app`.

---

## Bug 2: Relative Path Breaks from a Different Working Directory

### Step 1 — REPRODUCE

In [11]:
# Run from the PARENT directory: no cwd, so the script path includes debug_app/
# --data-dir keeps notes inside the project folder
result = run(
    [sys.executable, "debug_app/cli.py", "Buy groceries", "--data-dir", "debug_app/data"],
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
)

[stderr] Traceback (most recent call last):
  File "C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-runtime-failure-debugging\debug_app\cli.py", line 44, in <module>
    main()
  File "C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-runtime-failure-debugging\debug_app\cli.py", line 37, in main
    print(json.dumps(save_note(args.text, api_key, data_dir), indent=2))
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-runtime-failure-debugging\debug_app\core.py", line 15, in save_note
    config = load_config()
             ^^^^^^^^^^^^^
  File "C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-runtime-failure-debugging\debug_app\core.py", line 10, in load_config
    with open(config_path, encoding="utf-8") as f:
         ^^^^^^^

### Step 2 — CLASSIFY

`FileNotFoundError: ... 'config.json'` points to a **filesystem path** bug.

### Step 3 — ISOLATE

```python
config_path = Path("config.json")  # relative to CWD, not to the script
```

### Step 4 — FIX

Anchor the path to the script's own location with `__file__`.

In [12]:
%%writefile debug_app/core.py
"""Core logic — Bug 2 FIXED (anchored path)."""
import json
from pathlib import Path
from datetime import datetime, timezone

# Anchor to the directory containing THIS file, not the caller's cwd
_THIS_DIR = Path(__file__).resolve().parent


def load_config():
    config_path = _THIS_DIR / "config.json"
    with open(config_path, encoding="utf-8") as f:
        return json.load(f)


def save_note(text: str, api_key: str, data_dir: Path) -> dict:
    config = load_config()
    if len(text) > config["max_note_length"]:
        raise ValueError(f"Note exceeds {config['max_note_length']} chars")

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    filename = f"note_{timestamp}.txt"
    (data_dir / filename).write_text(text, encoding="utf-8")

    return {"status": "saved", "file": filename, "app": config["app_name"]}

Overwriting debug_app/core.py


### Step 5 — VERIFY

In [13]:
# Same command as the failing run, from outside the project: now it works
run(
    [sys.executable, "debug_app/cli.py", "Buy groceries", "--data-dir", "debug_app/data"],
    env={**clean_env, "NOTE_API_KEY": "sk-test-123"},
)

[stdout] {
  "status": "saved",
  "file": "note_20261018_202215.txt",
  "app": "NoteKeeper"
}
[exit code] 0


CompletedProcess(args=['C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe', 'debug_app/cli.py', 'Buy groceries', '--data-dir', 'debug_app/data'], returncode=0, stdout='{\n  "status": "saved",\n  "file": "note_20261018_202215.txt",\n  "app": "NoteKeeper"\n}\n', stderr='')

Anchor file paths to the script with `Path(__file__).resolve().parent`, not to the caller's working directory. Next: a port conflict.

---

## Bug 3: Port Conflict

### Step 1 — REPRODUCE

In [14]:
import socket

# Occupy port 8000 to simulate another service already using it
blocker = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
blocker.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
blocker.bind(("127.0.0.1", 8000))
blocker.listen(1)
print("Port 8000 is now occupied.")

Port 8000 is now occupied.


In [15]:
# Try to start uvicorn on the occupied port
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api:app", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd="debug_app",
    env={**os.environ, "NOTE_API_KEY": "sk-test-123"},
)
time.sleep(3)

# A nonzero exit code means the server crashed at startup
if server_proc.poll() is not None:
    stderr_output = server_proc.stderr.read().decode()
    print("[stderr]", stderr_output[-500:])  # the tail shows 'address already in use'
    print(f"[exit code] {server_proc.returncode}")

### Step 2 — CLASSIFY

"Address already in use" means a **port conflict**.

### Step 3 — ISOLATE

Our code is fine. The bug is launching with a hardcoded port.

### Step 4 — FIX

Build a launcher that reads the port from layered config and falls back to a free port.

In [16]:
%%writefile debug_app/serve.py
"""Launcher with port-conflict detection — Bug 3 FIXED."""
import os
import sys
import socket


def is_port_free(port: int) -> bool:
    # If we can bind the port ourselves, nobody else is using it
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        try:
            s.bind(("127.0.0.1", port))
            return True
        except OSError:
            return False


def pick_port(preferred: int, alternatives: list[int]) -> int:
    if is_port_free(preferred):
        return preferred
    for port in alternatives:
        if is_port_free(port):
            print(f"warning: port {preferred} in use, using {port}", file=sys.stderr)
            return port
    print("error: no free port found.", file=sys.stderr)
    sys.exit(1)


if __name__ == "__main__":
    import uvicorn

    # Layered config: APP_PORT env var → default 8000
    preferred = int(os.environ.get("APP_PORT", "8000"))
    port = pick_port(preferred, [8001])
    uvicorn.run("api:app", host="127.0.0.1", port=port)

Writing debug_app/serve.py


### Step 5 — VERIFY

> 🧩 **Http Endpoint Contract.** Watch for the request we send to `/note` and the status and JSON that come back.

In [17]:
import httpx

# Port 8000 is still occupied, so the launcher should fall back to 8001
server_proc = subprocess.Popen(
    [sys.executable, "serve.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd="debug_app",
    env={**os.environ, "NOTE_API_KEY": "sk-test-123"},
)
time.sleep(3)

resp = httpx.post("http://127.0.0.1:8001/note", json={"text": "Port conflict resolved!"})
print(f"Response: {resp.status_code} — {resp.json()}")

Response: 200 — {'status': 'saved', 'file': 'note_20261018_202223.txt', 'app': 'NoteKeeper'}


In [18]:
# Stop the server and release port 8000
server_proc.terminate()
server_proc.wait(timeout=5)
blocker.close()

---

## The Decision Procedure in Review

We fixed three bugs: a missing env var, a config path, and a port conflict. The same anchored-path technique also applies to default data directories such as `--data-dir data`.

In [19]:
summary_table = """
| Bug | Category | Root Cause | Fix Pattern |
|-----|----------|-----------|-------------|
| Missing env var | Environment | `os.environ[key]` crashes on missing key | Layered config: CLI arg → `os.environ.get()` → helpful error |
| Config path breaks | Filesystem path | `Path("config.json")` is relative to CWD | Anchor with `Path(__file__).resolve().parent / "config.json"` |
| Port already in use | Port conflict | Hardcoded port with no fallback | Check `is_port_free()`, fall back, make port configurable |

Same fix pattern applies to default data directories.
"""

md(summary_table)


| Bug | Category | Root Cause | Fix Pattern |
|-----|----------|-----------|-------------|
| Missing env var | Environment | `os.environ[key]` crashes on missing key | Layered config: CLI arg → `os.environ.get()` → helpful error |
| Config path breaks | Filesystem path | `Path("config.json")` is relative to CWD | Anchor with `Path(__file__).resolve().parent / "config.json"` |
| Port already in use | Port conflict | Hardcoded port with no fallback | Check `is_port_free()`, fall back, make port configurable |

Same fix pattern applies to default data directories.


In [20]:
import shutil

shutil.rmtree("debug_app", ignore_errors=True)
print("Cleaned up.")

Cleaned up.


## 🎯 Summary

### What You Built

You diagnosed and fixed three runtime-only bugs in a CLI + HTTP application using a five-step debugging procedure (Reproduce → Classify → Isolate → Fix → Verify).

### Key Takeaways

1. **Debugging decision procedure**: Reproduce, Classify, Isolate, Fix, Verify replaces random changes with a repeatable method
2. **Missing env vars**: `os.environ[key]` crashes at runtime; use layered configuration (CLI arg → `os.environ.get()` → clear error)
3. **Relative path bugs**: `Path("config.json")` resolves against the caller's working directory; anchor with `Path(__file__).resolve().parent`
4. **Port conflicts**: a hardcoded port fails once another process claims it; check with `socket.bind()` and fall back
5. **Tests don't catch runtime bugs**: fixtures set env vars and the working directory, so also run the app manually from a different directory